# Testing Notebook

In [1]:
import shared_utils

In [2]:
import _replica_utils

In [3]:
import gcsfs as fs
import geopandas as gpd
import numpy as np
import pandas as pd
from calitp_data_analysis import get_fs, utils
from calitp_data_analysis.sql import to_snakecase
# from siuba import *

fs = get_fs()

In [4]:
pd.set_option("display.max_columns", None)

In [5]:
gcs_path = "gs://calitp-analytics-data/data-analyses/big_data/STM/"

In [6]:
blk_grp_url = "CA_Census_blocks_w_Cities_centered.zip"

In [7]:
shape_data_name = "origins/California_hex7_layer.zip"
origins_name = "D7_8_11_12_replica-stm_regional_travel-06_25_26-trips_dataset.zip"

In [8]:
df = to_snakecase( pd.read_csv(f"{gcs_path}{origins_name}"))  

/tmp/ipykernel_2658/3138358181.py:1: DtypeWarning: Columns (58) have mixed types. Specify dtype option on import or set low_memory=False.
  df = to_snakecase( pd.read_csv(f"{gcs_path}{origins_name}"))


In [9]:
df_origins, df_dests = _replica_utils.prep_replica_data_w_shp(df, shape_data_name, 'origin_custom_id', 'destination_custom_id')

In [10]:
df_origins.sample()

,activity_id,origin_bgrp_2020,origin_trct_2020,origin_cty_2020,origin_st_2020,destination_bgrp_2020,destination_trct_2020,destination_cty_2020,destination_st_2020,primary_mode,trip_purpose,previous_trip_purpose,trip_start_time,trip_end_time,trip_duration_minutes,trip_distance_miles,vehicle_type,vehicle_fuel_type,transit_submode,transit_agency,transit_route,origin_land_use,origin_building_use,destination_land_use,destination_building_use,trip_taker_person_id,trip_taker_household_id,trip_taker_age,trip_taker_sex,trip_taker_race_ethnicity,trip_taker_employment_status,trip_taker_wfh,trip_taker_individual_income,trip_taker_commute_mode,trip_taker_household_size,trip_taker_household_income,trip_taker_available_vehicles,trip_taker_resident_type,trip_taker_industry,trip_taker_building_type,trip_taker_school_grade_attending,trip_taker_education,trip_taker_tenure,trip_taker_language,trip_taker_home_bgrp_2020,trip_taker_home_trct_2020,trip_taker_home_cty_2020,trip_taker_home_st_2020,trip_taker_work_bgrp_2020,trip_taker_work_trct_2020,trip_taker_work_cty_2020,trip_taker_work_st_2020,origin_custom,destination_custom,origin_custom_id,origin_bgrp_fips_2020,origin_custom_lng,origin_custom_lat,destination_bgrp_fips_2020,destination_custom_id,destination_custom_lat,destination_custom_lng,origin_geometry
78284,7988161174829495007,"1 (Tract 6201.01, Los Angeles, CA)","6201.01 (Los Angeles, CA)","Los Angeles County, CA",California,"1 (Tract 83.61, San Diego, CA)","83.61 (San Diego, CA)","San Diego County, CA",California,private_auto,shop,home,08:23:00,10:05:40,102,112.2,unknown_vehicle_type,electric,NaN,NaN,NaN,single_family,single_family,mixed_use,retail,9934288415228259263,17947204029998783816,56.0,male,asian_not_hispanic_or_latino,employed,in_person,77687.0,private_auto,3,210224.0,three_plus,core,naics31_33,single_family,not_attending_school,some_college,owner,asian_pacific,"1 (Tract 6201.01, Los Angeles, CA)","6201.01 (Los Angeles, CA)","Los Angeles County, CA",California,"1 (Tract 6025.13, Los Angeles, CA)","6025.13 (Los Angeles, CA)","Los Angeles County, CA",California,8729a565bffffff,8729a401dffffff,608718595349807100,60376201011,-118.4119,33.9337,060730083611,6.087185e+17,32.8729,-117.2362,"POLYGON ((-118.40364 33.92238, -118.41978 33.9..."


In [11]:
# with get_fs().open(f"{gcs_path}{place_data}") as f:
#         plc = to_snakecase(gpd.read_file(f))

In [12]:
# plc.sample()

In [13]:
# plc.columns

In [14]:
# plc = plc[['objectid', 'state', 'geoid', 'county', 'tract', 'blkgrp', 'name', 'basename', 'cdtfa_coun', 'city_name', 'geometry']]

In [15]:
# ### add in a period and ", CA" to match the format in the replica data 
# plc['tract'] = plc['tract'].str[:-2] + '.' + plc['tract'].str[-2:]
# plc['county_name'] = plc['cdtfa_coun'] + ", CA"


In [16]:
# plc['city_name'] = plc['city_name'].fillna('Unincorporated')

In [17]:


# ### Format the block group column by removing trailing and leading 0s
# ### Replica's data format is different for tract numbers
# ### Ex: 00111.00 vs 111
# condition = plc['tract'].str.endswith(".00")
# plc['tract'] = np.where(condition, plc['tract'].str.replace(".00", ""), plc['tract'])

# plc['corrected_tract'] = plc['tract'].str.lstrip('00')
# plc['corrected_tract'] = plc['tract'].str.lstrip('0')

In [18]:
# plc['city_county'] = plc['city_name'] + ', ' + plc['cdtfa_coun']

In [19]:
# plc.sample(10)

In [20]:
def prep_place_data(place_data_df, county_name_col, tract_col, city_name_col, state):

    ### add in a period and ", CA" to match the format in the replica data 
    place_data_df[tract_col] = place_data_df[tract_col].str[:-2] + '.' + place_data_df['tract'].str[-2:]
    place_data_df['county_name'] = place_data_df[[county_name_col]] + f", {state}"

    ### replace the na locations
    place_data_df[city_name_col] = place_data_df[city_name_col].fillna('Unincorporated')

    ### Format the block group column by removing trailing and leading 0s
    ### Replica's data format is different for tract numbers
    ### Ex: 00111.00 vs 111
    condition = place_data_df[tract_col].str.endswith(".00")
    place_data_df[tract_col] = np.where(condition, place_data_df[tract_col].str.replace(".00", ""), place_data_df['tract'])
    
    place_data_df['corrected_tract'] = place_data_df[tract_col].str.lstrip('00')
    place_data_df['corrected_tract'] = place_data_df[tract_col].str.lstrip('0')
    
    ### add together the county and city names, helpful for thing like "Unincorporated, County"
    place_data_df['city_county'] = place_data_df[city_name_col] + ', ' + place_data_df[county_name_col]

    return place_data_df

In [21]:
def read_and_prep_place_data(ca_place_path, nv_place_path,):
    
    ## read in the place data for CA
    with get_fs().open(f"{gcs_path}{ca_place_path}") as f:
        ca_plc = to_snakecase(gpd.read_file(f))
    ## susbet data
    ca_plc = ca_plc[['objectid', 'state', 'geoid', 'county', 'tract', 'blkgrp', 'name', 'basename', 'cdtfa_coun', 'city_name', 'geometry']]
    ## format
    ca = prep_place_data(ca_plc, county_name_col='cdtfa_coun', tract_col='tract', city_name_col='city_name', state="CA")

    ### repreat for NV
    with get_fs().open(f"{gcs_path}{nv_place_path}") as f:
        nv_plc = to_snakecase(gpd.read_file(f))
    nv_plc['countyname'] = nv_plc['countyname'] + " County"
    nv = prep_place_data(nv_plc, county_name_col='countyname', tract_col='tract', city_name_col='city_name', state="NV")

    nv_subset = nv[['corrected_tract', 'blkgrp','county_name', 'city_county']] 
    ca_subset = ca[['corrected_tract', 'blkgrp','county_name', 'city_county']]

    places = pd.concat([ca_subset, nv_subset], axis=0, ignore_index=True)

    return places

In [22]:
def add_cities_to_origin_dest(df, ca_place_path, nv_place_path, origin_county_col, orgin_tract_col, origin_blkgrp_col, dest_county_col, dest_tract_col, dest_blkgrp_col):

    places = read_and_prep_place_data(ca_place_path, nv_place_path)

    ### set up replica data by extracting just the tract number and blockgroup number 
    df['origin_tract'] = df[orgin_tract_col].str.split(' (', regex=False).str[0]
    df['origin_blkgrp'] = df[origin_blkgrp_col].str.split(' (', regex=False).str[0]
    
    df['dest_tract'] = df[dest_tract_col].str.split(' (', regex=False).str[0]
    df['dest_blkgrp'] = df[dest_blkgrp_col].str.split(' (', regex=False).str[0]

    ### merge together! 
    ### first merge origins to get the origin city and then merge the destinations to get destination city

    df2 = pd.merge(
        df, 
        places[['county_name', 'corrected_tract', 'blkgrp', 'city_county']],  
        left_on=[origin_county_col, 'origin_tract', 'origin_blkgrp'], 
        right_on=['county_name', 'corrected_tract', 'blkgrp'],
        how='left'
    )

    ### drop the blk_grps columms for the second merge and rename city col to distinguish
    df2 = df2.drop(columns=['county_name', 'corrected_tract', 'blkgrp'])
    df2 = df2.rename(columns={"city_county":"origin_city"})
    
    ### and repeat
    df2 = pd.merge(
        df2, 
       places[['county_name', 'corrected_tract', 'blkgrp', 'city_county']], 
        left_on=[dest_county_col, 'dest_tract', 'dest_blkgrp'], 
        right_on=['county_name', 'corrected_tract', 'blkgrp'],
        how='left'
    )
    
    df2 = df2.drop(columns=['county_name', 'corrected_tract', 'blkgrp'])
    df2 = df2.rename(columns={"city_county":"dest_city"})

    df2['dest_city'] = df2['dest_city'].fillna('Out of Region')

    return df2

In [23]:
ca_place_data = "CA_Census_Block_with_Places.zip"

In [24]:
nv_place_data = "NV_Census_Block_with_Places.zip"

In [25]:
df2 = add_cities_to_origin_dest(df_origins, ca_place_data, nv_place_data, "origin_cty_2020", "origin_trct_2020", "origin_bgrp_2020", "destination_cty_2020", "destination_trct_2020", "destination_bgrp_2020")

In [26]:
# ## read in the data
# with get_fs().open(f"{gcs_path}{ca_place_data}") as f:
#     ca_plc = to_snakecase(gpd.read_file(f))

# ca_plc = ca_plc[['objectid', 'state', 'geoid', 'county', 'tract', 'blkgrp', 'name', 'basename', 'cdtfa_coun', 'city_name', 'geometry']]

In [27]:
# test_ca = prep_place_data(ca_plc, county_name_col='cdtfa_coun', tract_col='tract', city_name_col='city_name', state="CA")

In [28]:
# test_ca.sample()

In [29]:
# with get_fs().open(f"{gcs_path}{nv_place_data}") as f:
#     nv_plc = to_snakecase(gpd.read_file(f))

In [30]:
# nv_plc['countyname'] = nv_plc['countyname'] + " County"

In [31]:
# test_nv = prep_place_data(nv_plc, county_name_col='countyname', tract_col='tract', city_name_col='city_name', state="NV")

In [32]:
# test_nv.sample()

In [33]:
# nv_subset = test_nv[['corrected_tract', 'blkgrp','county_name', 'city_county']] 
# ca_subset = test_ca[['corrected_tract', 'blkgrp','county_name', 'city_county']]

In [34]:
# places = pd.concat([ca_subset, nv_subset], axis=0, ignore_index=True)

In [35]:
# places

In [36]:
# df_origins.sample()

In [37]:
# ### set up replica data by extracting just the tract number and blockgroup number 
# df_origins['origin_tract'] = df_origins['origin_trct_2020'].str.split(' (', regex=False).str[0]
# df_origins['origin_blkgrp'] = df_origins['origin_bgrp_2020'].str.split(' (', regex=False).str[0]
    
# df_origins['dest_tract'] = df_origins['destination_trct_2020'].str.split(' (', regex=False).str[0]
# df_origins['dest_blkgrp'] = df_origins['destination_bgrp_2020'].str.split(' (', regex=False).str[0]



In [38]:
# df_origins.sample()

In [39]:
# ### merge together! 
# ### first merge origins to get the origin city and then merge the destinations to get destination city

# df2 = pd.merge(
#         df_origins, 
#         places[['county_name', 'corrected_tract', 'blkgrp', 'city_county']], 
#         left_on=['origin_cty_2020', 'origin_tract', 'origin_blkgrp'], 
#         right_on=['county_name', 'corrected_tract', 'blkgrp'],
#         how='left'
#     )


In [40]:
# ### drop the blk_grps columms for the second merge and rename city col to distinguish
# df2 = df2.drop(columns=['county_name', 'corrected_tract', 'blkgrp'])
# df2 = df2.rename(columns={"city_county":"origin_city"})
    

In [41]:
# ### and repeat
# df2 = pd.merge(
#         df2, 
#         places[['county_name', 'corrected_tract', 'blkgrp', 'city_county']], 
#         left_on=['destination_cty_2020', 'dest_tract', 'dest_blkgrp'], 
#         right_on=['county_name', 'corrected_tract', 'blkgrp'],
#         how='left'
#     )

In [42]:
# df2 = df2.drop(columns=['county_name', 'corrected_tract', 'blkgrp'])
# df2 = df2.rename(columns={"city_county":"dest_city"})

In [43]:
# df2['dest_city'] = df2['dest_city'].fillna('Out of Region')

In [44]:
# df2[df2["destination_cty_2020"].str.contains("NV")]

In [45]:
#### Checking specific city that did not merge....

In [46]:
places = read_and_prep_place_data(ca_place_data, nv_place_data,)

In [47]:
with get_fs().open(f"{gcs_path}{ca_place_data}") as f:
        ca_plc = to_snakecase(gpd.read_file(f))

In [48]:
places.sample()

,corrected_tract,blkgrp,county_name,city_county
9579,13.04,1,"Ventura County, CA","San Buenaventura (Ventura), Ventura County"


In [49]:
ca_plc[ca_plc['tract']=="980033"]

,objectid,join_count,target_fid,mtfcc,oid,geoid,state,county,tract,blkgrp,basename,name,lsadc,funcstat,arealand,areawater,ur,centlat,centlon,intptlat,intptlon,hu100,pop100,fid,cdtfa_coun,cdt_county,geoidfq,city_name,namelsad,shape_length,shape_area,stgeometry_length,stgeometry_area,geometry
7193,7194,0,70509,G5030,2087015903026004,060379800331,06,037,980033,1,1,Block Group 1,BG,S,12416589,16221301,M,+33.7471177,-118.2168578,+33.7443155,-118.2203101,10,89,NaN,None,None,None,None,None,NaN,NaN,28538.574262,4.152818e+07,"POLYGON ((-13163414.682 3996072.533, -13163383..."


In [50]:
places[places['corrected_tract']=="9800.33"]

,corrected_tract,blkgrp,county_name,city_county
7193,9800.33,1,NaN,NaN


In [51]:
# places.loc[(places['corrected_tract'] == '9800.33') & (places['county_name'].isna()), 'county_name'] = "Long Beach, "

In [52]:
# places[places["city_county"].str.contains("Long Beach")]

In [53]:
# ca_plc[ca_plc["tract"].str.contains("9901")]

In [54]:
# places[places["county_name"].isna()]

In [55]:
### find the top OD pairs

In [56]:
df2.sample()

,activity_id,origin_bgrp_2020,origin_trct_2020,origin_cty_2020,origin_st_2020,destination_bgrp_2020,destination_trct_2020,destination_cty_2020,destination_st_2020,primary_mode,trip_purpose,previous_trip_purpose,trip_start_time,trip_end_time,trip_duration_minutes,trip_distance_miles,vehicle_type,vehicle_fuel_type,transit_submode,transit_agency,transit_route,origin_land_use,origin_building_use,destination_land_use,destination_building_use,trip_taker_person_id,trip_taker_household_id,trip_taker_age,trip_taker_sex,trip_taker_race_ethnicity,trip_taker_employment_status,trip_taker_wfh,trip_taker_individual_income,trip_taker_commute_mode,trip_taker_household_size,trip_taker_household_income,trip_taker_available_vehicles,trip_taker_resident_type,trip_taker_industry,trip_taker_building_type,trip_taker_school_grade_attending,trip_taker_education,trip_taker_tenure,trip_taker_language,trip_taker_home_bgrp_2020,trip_taker_home_trct_2020,trip_taker_home_cty_2020,trip_taker_home_st_2020,trip_taker_work_bgrp_2020,trip_taker_work_trct_2020,trip_taker_work_cty_2020,trip_taker_work_st_2020,origin_custom,destination_custom,origin_custom_id,origin_bgrp_fips_2020,origin_custom_lng,origin_custom_lat,destination_bgrp_fips_2020,destination_custom_id,destination_custom_lat,destination_custom_lng,origin_geometry,origin_tract,origin_blkgrp,dest_tract,dest_blkgrp,origin_city,dest_city
72025,14281135760095867863,"2 (Tract 5759.02, Los Angeles, CA)","5759.02 (Los Angeles, CA)","Los Angeles County, CA",California,"3 (Tract 9005.10, Los Angeles, CA)","9005.10 (Los Angeles, CA)","Los Angeles County, CA",California,private_auto,home,eat,17:59:00,20:13:21,134,95.7,unknown_vehicle_type,other_non_bev,NaN,NaN,NaN,retail,retail,single_family,single_family,5201093835786092127,5889954983441411339,31.0,male,hispanic_or_latino_origin,employed,in_person,86955.0,private_auto,4,86955.0,two,core,not_working,single_family,not_attending_school,bachelors_degree,owner,spanish,"3 (Tract 9005.10, Los Angeles, CA)","9005.10 (Los Angeles, CA)","Los Angeles County, CA",California,"2 (Tract 2913, Los Angeles, CA)","2913 (Los Angeles, CA)","Los Angeles County, CA",California,8729a5614ffffff,8729a1442ffffff,608718594158624800,60375759022,-118.1908,33.767,060379005103,6.087183e+17,34.6863,-118.0853,"POLYGON ((-118.18255 33.75568, -118.19868 33.7...",5759.02,2,9005.10,3,"Long Beach, Los Angeles County","Lancaster, Los Angeles County"


In [57]:
### just at the city level, not granular enough
# (df2.groupby(['origin_city', 'dest_city'])['activity_id'].nunique()).reset_index().sort_values('activity_id')

In [58]:
def setup_od_map(df, origin_customid_col, dest_customid_col, origin_city_col, dest_city_col, origin_lat, origin_long, dest_lat, dest_long, n_rows):
    
    
    top_od = (df.groupby([origin_customid_col, dest_customid_col, origin_city_col, dest_city_col, origin_lat, origin_long, dest_lat, dest_long])['activity_id'].nunique()).reset_index().sort_values('activity_id', ascending=False).head(n_rows)

    lines = [
        LineString([(lon_s, lat_s), (lon_e, lat_e)]) 
        for lon_s, lat_s, lon_e, lat_e 
        in zip(top_od[origin_long], top_od[origin_lat], top_od[dest_long], top_od[dest_lat])
        ]
    
    od_pairs = gpd.GeoDataFrame(top_od, geometry=lines, crs="EPSG:4326") 

    od_pairs = od_pairs.to_crs(epsg=3310)

    return od_pairs

In [59]:
# ### adding in custom geo id to get the hex cells that have the most pairs... will also attempt with 
# top_od = (df2.groupby(['origin_custom_id', 'destination_custom_id','origin_city', 'dest_city','origin_custom_lng','origin_custom_lat', 'destination_custom_lat', 'destination_custom_lng'])['activity_id'].nunique()).reset_index().sort_values('activity_id', ascending=False).head(70)

In [60]:
# top_od.sample()

In [61]:
#### trying this approach
### https://plotly.com/python/lines-on-maps/

In [62]:
# import plotly.graph_objects as go

In [63]:
# import matplotlib.pyplot as plt
from shapely.geometry import LineString


In [64]:
# lines = [
#     LineString([(lon_s, lat_s), (lon_e, lat_e)]) 
#     for lon_s, lat_s, lon_e, lat_e 
#     in zip(top_od['origin_custom_lng'], top_od['origin_custom_lat'], top_od['destination_custom_lng'], top_od['destination_custom_lat'])
# ]

# gdf_lines = gpd.GeoDataFrame(top_od, geometry=lines, crs="EPSG:4326")

In [65]:
# gdf_lines = gdf_lines.to_crs(epsg=3310)


In [66]:
df2.sample()

,activity_id,origin_bgrp_2020,origin_trct_2020,origin_cty_2020,origin_st_2020,destination_bgrp_2020,destination_trct_2020,destination_cty_2020,destination_st_2020,primary_mode,trip_purpose,previous_trip_purpose,trip_start_time,trip_end_time,trip_duration_minutes,trip_distance_miles,vehicle_type,vehicle_fuel_type,transit_submode,transit_agency,transit_route,origin_land_use,origin_building_use,destination_land_use,destination_building_use,trip_taker_person_id,trip_taker_household_id,trip_taker_age,trip_taker_sex,trip_taker_race_ethnicity,trip_taker_employment_status,trip_taker_wfh,trip_taker_individual_income,trip_taker_commute_mode,trip_taker_household_size,trip_taker_household_income,trip_taker_available_vehicles,trip_taker_resident_type,trip_taker_industry,trip_taker_building_type,trip_taker_school_grade_attending,trip_taker_education,trip_taker_tenure,trip_taker_language,trip_taker_home_bgrp_2020,trip_taker_home_trct_2020,trip_taker_home_cty_2020,trip_taker_home_st_2020,trip_taker_work_bgrp_2020,trip_taker_work_trct_2020,trip_taker_work_cty_2020,trip_taker_work_st_2020,origin_custom,destination_custom,origin_custom_id,origin_bgrp_fips_2020,origin_custom_lng,origin_custom_lat,destination_bgrp_fips_2020,destination_custom_id,destination_custom_lat,destination_custom_lng,origin_geometry,origin_tract,origin_blkgrp,dest_tract,dest_blkgrp,origin_city,dest_city
110537,7756649553451233145,"1 (Tract 9800.13, Los Angeles, CA)","9800.13 (Los Angeles, CA)","Los Angeles County, CA",California,"2 (Tract 38.02, Tulare, CA)","38.02 (Tulare, CA)","Tulare County, CA",California,private_auto,shop,work,11:30:00,14:27:20,177,168.9,unknown_vehicle_type,electric,NaN,NaN,NaN,office,office,retail,retail,6491313201772352935,17065089993885521215,43.0,male,hispanic_or_latino_origin,employed,in_person,42737.0,private_auto,5,42737.0,three_plus,core,naics56,multiple_units,not_attending_school,high_school,renter,spanish,"4 (Tract 4615.02, Los Angeles, CA)","4615.02 (Los Angeles, CA)","Los Angeles County, CA",California,"1 (Tract 9800.13, Los Angeles, CA)","9800.13 (Los Angeles, CA)","Los Angeles County, CA",California,8729a565affffff,8729a85a4ffffff,608718595333029900,60379800131,-118.3875,33.9229,061070038022,6.087188e+17,36.0871,-119.0233,"POLYGON ((-118.37921 33.91162, -118.39535 33.9...",9800.13,1,38.02,2,"El Segundo, Los Angeles County","Porterville, Tulare County"


In [67]:
# gdf_lines = _replica_utils.setup_od_map(df2, 'origin_custom_id', 'destination_custom_id','origin_city', 'dest_city', 'origin_custom_lat', 'origin_custom_lng', 'destination_custom_lat', 'destination_custom_lng', 100)

In [68]:
# gdf_lines

In [69]:
# gdf_lines.explore(column="activity_id",  cmap='YlOrRd', tiles='CartoDB positron')